# Week 87, Day 4: Repository, CI and skeleton

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w087d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import re

messages = ["feat(routing): find round walks near a target time",
            "fix: count climb only once on loops",
            "docs: explain Naismith's rule",
            "feat!: store heights in metres, not feet",
            "Updated stuff"]
pattern = re.compile(r"(\w+)(\((.+)\))?(!)?: (.+)")
for m in messages:
    match = pattern.match(m)
    print(match.group(1), match.group(4) == "!", match.group(3)) if match else print("not conventional:", m)

In [ ]:
from fnmatch import fnmatch

patterns = ["__pycache__/", "*.pyc", ".env", "/data/", "*.db", "!tests/fixtures/*.db"]
for path in ["src/waymark/__pycache__/routing.cpython-312.pyc", "data/paths.csv", "docs/data/notes.md",
             "waymark.db", "tests/fixtures/small.db", ".env", "src/waymark/routing.py"]:
    print(f"{path:48}", [p for p in patterns if fnmatch(path.split("/")[-1], p.strip("/!").split("/")[-1])])

> **Key idea.** Lay out the repository with `pyproject.toml`, README, LICENSE, CHANGELOG, `.gitignore`, `src/package/`, `tests/`, `docs/`, `tools/` and a CI workflow. Keep secrets, generated files, environments, local databases and large data out, with `.gitignore` written before the first commit (last matching pattern wins; `/` anchors, trailing `/` means folders, `!` re-includes). Commit small, working changes with Conventional Commit messages (`type(scope)!: description`), from which the next version (breaking: major; feat: minor; fix or perf: patch) and the changelog follow. CI installs, lints, tests and checks the architecture on every push, fastest first, across a Python matrix; `main` is protected and always works. Start with a walking skeleton: every layer connected, one test, green CI.

### Exercise: Scaffolding a project

Write `skeleton(name, description, modules)`: a dict from relative path to file contents for a new project: `pyproject.toml` (exactly as in the checks), `README.md` (`# NAME`, a blank line, the description, a blank line, then `## Installation`, `## Usage` and `## Development` headings each followed by a blank line and `To do.`, with blank lines between sections), `.gitignore` (the lines of `IGNORE`, given, one per line, ending with a newline), `src/NAME/__init__.py` (`__version__ = "0.1.0"` and a newline), one `src/NAME/MODULE.py` per module containing a one-line docstring `"""The MODULE layer of NAME."""` and a newline, and `tests/test_smoke.py` (as in the checks). Then `write_tree(files, root)`: write them all under the folder `root` (creating folders as needed) and return the sorted list of paths written, relative to `root`, with `/` separators.

In [ ]:
from pathlib import Path

IGNORE = ["__pycache__/", "*.pyc", ".venv/", ".env", "*.db", "build/", "dist/", "*.egg-info/", ".coverage"]

def skeleton(name, description, modules):
    ___

def write_tree(files, root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\n\nPYPROJECT = '''[project]\nname = \"waymark\"\nversion = \"0.1.0\"\ndescription = \"Plan walks on local footpaths.\"\nrequires-python = \">=3.12\"\ndependencies = []\n\n[project.optional-dependencies]\ndev = [\"pytest\", \"ruff\"]\n\n[tool.pytest.ini_options]\ntestpaths = [\"tests\"]\n'''\n\nSMOKE = '''import waymark\n\n\ndef test_version():\n    assert waymark.__version__ == \"0.1.0\"\n'''\n\ndef test_skeleton():\n    \"the standard files\"\n    files = skeleton(\"waymark\", \"Plan walks on local footpaths.\", [\"domain\", \"storage\", \"services\", \"cli\"])\n    assert files[\"pyproject.toml\"] == PYPROJECT\n    assert files[\"tests/test_smoke.py\"] == SMOKE\n    assert files[\"README.md\"] == (\"# waymark\\n\\nPlan walks on local footpaths.\\n\\n## Installation\\n\\nTo do.\\n\\n\"\n                                  \"## Usage\\n\\nTo do.\\n\\n## Development\\n\\nTo do.\\n\")\n    assert files[\".gitignore\"].splitlines()[0] == \"__pycache__/\" and files[\".gitignore\"].endswith(\".coverage\\n\")\n    assert files[\"src/waymark/storage.py\"] == '\"\"\"The storage layer of waymark.\"\"\"\\n'\n    assert files[\"src/waymark/__init__.py\"] == '__version__ = \"0.1.0\"\\n'\n    paths = write_tree(files, \"out/waymark\")\n    assert paths == sorted(files) and len(paths) == 9\n    assert open(\"out/waymark/src/waymark/cli.py\").read() == '\"\"\"The cli layer of waymark.\"\"\"\\n'\n    assert os.path.isdir(\"out/waymark/tests\")")

### Exercise: Ignore rules

Write `ignored(path, patterns)` for a file path relative to the repository root (with `/` separators) and a list of `.gitignore` patterns. A pattern matches if it matches the file's name **or any of its folders**, with these rules: a trailing `/` means it only matches folders (so not the final part of the path); a leading `/` means it only matches the **first** part of the path; otherwise it may match any part; wildcards follow `fnmatch`; a pattern containing a `/` in the middle (like `tests/fixtures/*.db`) is matched against the whole path. A leading `!` makes a match mean "not ignored". The **last** pattern that matches decides; if none does, the file isn't ignored.

In [ ]:
from fnmatch import fnmatch

def ignored(path, patterns):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PATTERNS = [\"__pycache__/\", \"*.pyc\", \".env\", \"/data/\", \"*.db\", \"!tests/fixtures/*.db\", \"build/\"]\n\ndef test_ignore():\n    \"names, folders, anchors and exceptions\"\n    cases = {\"src/waymark/__pycache__/routing.cpython-312.pyc\": True, \"data/paths.csv\": True,\n             \"docs/data/notes.md\": False, \"waymark.db\": True, \"tests/fixtures/small.db\": False,\n             \".env\": True, \"config/.env\": True, \"src/waymark/routing.py\": False,\n             \"build\": False, \"build/lib/waymark.py\": True, \"src/build.py\": False}\n    assert {p: ignored(p, PATTERNS) for p in cases} == cases\n    assert ignored(\"notes.log\", [\"*.log\", \"!notes.log\"]) is False and ignored(\"notes.log\", [\"!notes.log\", \"*.log\"]) is True")

### Exercise: Conventional commits and the next version

Write `parse_commit(message)`: for a message whose first line is `type(scope)!: description` (scope and `!` optional; type lower-case letters), return a dict with `type`, `scope` (or `None`), `breaking` (`True` if there's a `!` or any later line starts with `BREAKING CHANGE:`) and `description`; raise `ValueError("not a conventional commit")` otherwise, or if the type isn't in `TYPES` (given). Then `next_version(current, messages)`: the next semantic version: a major bump if any commit is breaking, else minor if any is a `feat`, else patch if any is a `fix` or `perf`, else `current` unchanged.

In [ ]:
import re

TYPES = {"feat", "fix", "docs", "test", "refactor", "perf", "build", "ci", "chore", "style"}

def parse_commit(message):
    ___

def next_version(current, messages):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_commits():\n    \"type, scope, breaking, description\"\n    assert parse_commit(\"feat(routing): find round walks near a target time\") == {\n        \"type\": \"feat\", \"scope\": \"routing\", \"breaking\": False, \"description\": \"find round walks near a target time\"}\n    assert parse_commit(\"feat!: store heights in metres\")[\"breaking\"] is True\n    assert parse_commit(\"refactor: split storage\\n\\nBREAKING CHANGE: load_paths now needs a connection\")[\"breaking\"] is True\n    for bad in (\"Updated stuff\", \"feature: x\", \"Fix: y\", \"fix:no space\"):\n        try:\n            parse_commit(bad)\n        except ValueError as e:\n            assert str(e) == \"not a conventional commit\"\n        else:\n            raise AssertionError(bad)\n\ndef test_versions():\n    \"breaking, feature, fix, nothing\"\n    assert next_version(\"1.4.2\", [\"fix: a\", \"docs: b\"]) == \"1.4.3\"\n    assert next_version(\"1.4.2\", [\"fix: a\", \"feat: b\"]) == \"1.5.0\"\n    assert next_version(\"1.4.2\", [\"feat: b\", \"fix(api)!: c\"]) == \"2.0.0\"\n    assert next_version(\"1.4.2\", [\"docs: b\", \"ci: c\"]) == \"1.4.2\"\n    assert next_version(\"0.9.1\", [\"perf: faster search\"]) == \"0.9.2\"")

### Exercise: A changelog from commits

Using the given `parse_commit`, write `changelog_entry(version, date, messages)`: a Markdown section `## [VERSION] - DATE`, then, for each group that has entries, a blank line, a `### Group` heading and one `- ` line per commit in the order given: **Added** for `feat`, **Fixed** for `fix`, **Changed** for `perf` and `refactor` (other types are left out). Each line is the description, with `**scope:** ` before it if there's a scope, and ` (breaking)` after it if it's breaking. End with a newline.

In [ ]:
import re

TYPES = {"feat", "fix", "docs", "test", "refactor", "perf", "build", "ci", "chore", "style"}
HEADER = re.compile(r"(?P<type>[a-z]+)(?:\((?P<scope>[^)]+)\))?(?P<bang>!)?: (?P<description>\S.*)$")

def parse_commit(message):
    first, *rest = message.split("\n")
    m = HEADER.match(first)
    if not m or m.group("type") not in TYPES:
        raise ValueError("not a conventional commit")
    return {"type": m.group("type"), "scope": m.group("scope"),
            "breaking": bool(m.group("bang")) or any(l.startswith("BREAKING CHANGE:") for l in rest),
            "description": m.group("description")}

def changelog_entry(version, date, messages):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_changelog():\n    \"grouped and readable\"\n    msgs = [\"feat(routing): find round walks near a target time\", \"docs: explain Naismith's rule\",\n            \"fix: count climb only once on loops\", \"feat!: store heights in metres\", \"perf(routing): cache neighbours\",\n            \"ci: test on 3.13\"]\n    assert changelog_entry(\"2.0.0\", \"2026-11-11\", msgs) == \"\"\"## [2.0.0] - 2026-11-11\n\n### Added\n- **routing:** find round walks near a target time\n- store heights in metres (breaking)\n\n### Fixed\n- count climb only once on loops\n\n### Changed\n- **routing:** cache neighbours\n\"\"\"\n    assert changelog_entry(\"1.0.1\", \"2026-12-01\", [\"fix: typo in help\"]) == \"## [1.0.1] - 2026-12-01\\n\\n### Fixed\\n- typo in help\\n\"")

### Exercise: Writing the CI workflow

Write `ci_workflow(pythons, commands)`: the text of a GitHub Actions workflow exactly like the one in the lesson, with the matrix's `python-version` list built from `pythons` (each in double quotes, separated by `, `) and one `- run: COMMAND` step per command after the setup steps, in order. Indent with two spaces per level as in the lesson, and end with a newline.

In [ ]:
def ci_workflow(pythons, commands):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("EXPECTED = '''name: CI\non: [push, pull_request]\njobs:\n  test:\n    runs-on: ubuntu-latest\n    strategy:\n      matrix:\n        python-version: [\"3.12\", \"3.13\"]\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: ${{ matrix.python-version }}\n      - run: pip install -e \".[dev]\"\n      - run: ruff check .\n      - run: pytest\n      - run: python tools/check_architecture.py\n'''\n\ndef test_workflow():\n    \"checkout, Python, then the checks\"\n    cmds = ['pip install -e \".[dev]\"', \"ruff check .\", \"pytest\", \"python tools/check_architecture.py\"]\n    assert ci_workflow([\"3.12\", \"3.13\"], cmds) == EXPECTED\n    assert 'python-version: [\"3.13\"]' in ci_workflow([\"3.13\"], [\"pytest\"])")

### Exercise: Checking a README

Write `readme_problems(text)`: a list of problems in this order, each only if it applies: `"no title"` (the first non-blank line isn't a `# ` heading); `"missing section: NAME"` for each of `Installation`, `Usage` and `Development` with no `## ` heading containing that word (ignoring case); `"no example"` if there's no fenced code block (a line starting with three backticks); and `"no licence"` if neither `licence` nor `license` appears (ignoring case).

In [ ]:
import re

def readme_problems(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FENCE = \"`\" * 3\n\nGOOD = f\"\"\"# Waymark\n\nPlan walks on local footpaths, with times that include the climbing.\n\n## Installation\n\n    pip install waymark\n\n## Usage\n\n{FENCE}\n$ waymark plan Mill Ridge\nMill -> Weir -> Ridge: 6.5 km, 2 h 10 min\n{FENCE}\n\n## Development (tests and checks)\n\nInstall with dev extras and run pytest.\n\n## Licence\n\nMIT.\n\"\"\"\n\ndef test_readme():\n    \"the questions a README answers\"\n    assert readme_problems(GOOD) == []\n    assert readme_problems(\"Waymark\\n\\n## How to use\\n\") == [\"no title\", \"missing section: Installation\",\n                                                            \"missing section: Usage\", \"missing section: Development\",\n                                                            \"no example\", \"no licence\"]\n    assert readme_problems(GOOD.replace(\"## Usage\", \"## Using it\")) == [\"missing section: Usage\"]\n    assert readme_problems(\"\\n\\n# X\\n## installation\\n## USAGE\\n## development\\n\" + FENCE + \"\\nx\\n\" + FENCE + \"\\nLicense: MIT\\n\") == []")

### Bug hunt: Folders that don't exist yet

Writing `src/waymark/domain.py` fails with `FileNotFoundError` because the folders aren't there. Create the parent folders first, so it prints `['src/waymark/__init__.py', 'src/waymark/domain.py']`.

In [ ]:
from pathlib import Path

files = {"src/waymark/__init__.py": "", "src/waymark/domain.py": '"""Domain."""\n'}
for rel, text in files.items():
    Path("proj", rel).write_text(text)
print(sorted(str(p.relative_to("proj")).replace("\\", "/") for p in Path("proj").rglob("*.py")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"mkdir(parents=True)\"\n    expect_output(\"['src/waymark/__init__.py', 'src/waymark/domain.py']\")")

### Bug hunt: A breaking change released as a feature

The version logic checks for features before breaking changes, so a release with both is only a minor bump: it prints `1.5.0`. Check breaking changes first, so it prints `2.0.0`.

In [ ]:
def bump(current, kinds):
    major, minor, patch = map(int, current.split("."))
    if "feat" in kinds:
        return f"{major}.{minor + 1}.0"
    if "breaking" in kinds:
        return f"{major + 1}.0.0"
    if "fix" in kinds:
        return f"{major}.{minor}.{patch + 1}"
    return current

print(bump("1.4.2", {"feat", "breaking", "fix"}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the biggest change wins\"\n    expect_output(\"2.0.0\")")

### Bug hunt: A bump that forgot to reset

A minor release must reset the patch number, but this keeps it: it prints `1.5.2`. Reset it, so it prints `1.5.0`.

In [ ]:
major, minor, patch = 1, 4, 2
print(f"{major}.{minor + 1}.{patch}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"reset the parts to the right\"\n    expect_output(\"1.5.0\")")

### Puzzle: The steps of a CI job

Arrange the steps of Waymark's CI job, fastest checks first. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the next version after 2.3.1 for commits "fix: a" and "feat(cli): b"
q = ___   # True or False: ".env" belongs in the repository
r = ___   # the Conventional Commit type for a change that only makes code faster
s = ___   # the folder that holds GitHub Actions workflows
t = ___   # "walking skeleton" or "vertical slice": every layer connected, doing almost nothing

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"2.4.0\" and q is False and r == \"perf\", \"p, q and r\"\n    assert s.strip(\"/\") == \".github/workflows\" and t == \"walking skeleton\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: preparing a release

Write `prepare_release(current, messages, date, changelog)` using the given `parse_commit`, `next_version` and `changelog_entry`. If any message isn't conventional, raise `ValueError` listing them: `"not conventional: A; B"` (the first lines, in order). If the next version equals `current`, return `None` (nothing to release). Otherwise return a dict with `version`, `tag` (`"v"` plus the version), `entry` (the changelog entry) and `changelog`: the existing changelog text with the entry inserted **after** the first line (the `# Changelog` title) and its blank line, and before any earlier entries, followed by a blank line. Also add `summary`: `"VERSION: N commits (A added, F fixed, B breaking)"`.

In [ ]:
import re

TYPES = {"feat", "fix", "docs", "test", "refactor", "perf", "build", "ci", "chore", "style"}
HEADER = re.compile(r"(?P<type>[a-z]+)(?:\((?P<scope>[^)]+)\))?(?P<bang>!)?: (?P<description>\S.*)$")

def parse_commit(message):
    first, *rest = message.split("\n")
    m = HEADER.match(first)
    if not m or m.group("type") not in TYPES:
        raise ValueError("not a conventional commit")
    return {"type": m.group("type"), "scope": m.group("scope"),
            "breaking": bool(m.group("bang")) or any(l.startswith("BREAKING CHANGE:") for l in rest),
            "description": m.group("description")}

def next_version(current, messages):
    major, minor, patch = (int(p) for p in current.split("."))
    commits = [parse_commit(m) for m in messages]
    if any(c["breaking"] for c in commits):
        return f"{major + 1}.0.0"
    if any(c["type"] == "feat" for c in commits):
        return f"{major}.{minor + 1}.0"
    if any(c["type"] in ("fix", "perf") for c in commits):
        return f"{major}.{minor}.{patch + 1}"
    return current

def changelog_entry(version, date, messages):
    commits = [parse_commit(m) for m in messages]
    lines = [f"## [{version}] - {date}"]
    for title, types in [("Added", {"feat"}), ("Fixed", {"fix"}), ("Changed", {"perf", "refactor"})]:
        entries = [c for c in commits if c["type"] in types]
        if entries:
            lines += ["", f"### {title}"]
            lines += [f"- {'**' + c['scope'] + ':** ' if c['scope'] else ''}{c['description']}{' (breaking)' if c['breaking'] else ''}"
                      for c in entries]
    return "\n".join(lines) + "\n"

def prepare_release(current, messages, date, changelog):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("OLD = \"# Changelog\\n\\n## [0.1.0] - 2026-11-04\\n\\n### Added\\n- walking skeleton\\n\"\n\ndef test_release():\n    \"version, tag, changelog\"\n    msgs = [\"feat(routing): shortest routes\", \"fix: round times to the minute\", \"docs: README usage\", \"test: S1 scenarios\"]\n    r = prepare_release(\"0.1.0\", msgs, \"2026-11-11\", OLD)\n    assert r[\"version\"] == \"0.2.0\" and r[\"tag\"] == \"v0.2.0\"\n    assert r[\"changelog\"] == (\"# Changelog\\n\\n## [0.2.0] - 2026-11-11\\n\\n### Added\\n- **routing:** shortest routes\\n\\n\"\n                              \"### Fixed\\n- round times to the minute\\n\\n## [0.1.0] - 2026-11-04\\n\\n### Added\\n- walking skeleton\\n\")\n    assert r[\"summary\"] == \"0.2.0: 4 commits (1 added, 1 fixed, 0 breaking)\"\n    assert prepare_release(\"0.2.0\", [\"docs: typo\", \"ci: cache pip\"], \"2026-11-12\", r[\"changelog\"]) is None\n\ndef test_refusals():\n    \"only conventional history is released\"\n    try:\n        prepare_release(\"1.0.0\", [\"feat: ok\", \"Updated stuff\", \"wip\\n\\nmore\"], \"2026-11-11\", OLD)\n    except ValueError as e:\n        assert str(e) == \"not conventional: Updated stuff; wip\"\n    else:\n        raise AssertionError(\"should refuse\")\n    r = prepare_release(\"1.2.3\", [\"feat!: new map format\"], \"2026-11-20\", \"# Changelog\\n\\n\")\n    assert r[\"version\"] == \"2.0.0\" and r[\"summary\"] == \"2.0.0: 1 commits (1 added, 0 fixed, 1 breaking)\"\n    assert r[\"changelog\"] == \"# Changelog\\n\\n## [2.0.0] - 2026-11-20\\n\\n### Added\\n- new map format (breaking)\\n\\n\"")

## Warm-up

Each conventional message says what **kind** of change it is (`feat`, `fix`, `docs`), optionally where (`routing`), whether it **breaks** something (`!`), and what it does. That's enough for a program to decide that the next release is a new major version and to write its changelog. "Updated stuff" helps nobody, including you in five weeks.

## The layout

Waymark's repository, which is the shape to copy:

```
waymark/
├── pyproject.toml          name, version, dependencies, tool settings (Week 73)
├── README.md               what it is, how to install, use and develop it
├── LICENSE                 the licence text (MIT is a common, simple choice)
├── CHANGELOG.md            what changed in each version
├── .gitignore
├── .github/workflows/ci.yml
├── docs/                   spec.md, schema.sql, adr/, notes/
├── src/waymark/            the package: __init__.py, domain.py, routing.py, storage.py, services.py, api.py, cli.py
├── tests/                  test_domain.py, test_routing.py, ...
└── tools/                  check_architecture.py and other scripts
```

The **src layout** (the package inside `src/`) means tests run against the installed package (`pip install -e ".[dev]"`), not whatever happens to be in the current folder, which catches packaging mistakes early. `pyproject.toml` also holds settings for tools such as pytest and the linter, so there's one place to look.

The **README** answers, in order: what is this, and who is it for (yesterday's pitch); how do I install it; how do I use it (a real example, with output); how do I develop it (install with dev extras, run the tests, the architecture check); and the licence.

## What stays out

Some files never belong in a repository: **secrets** (`.env` files, keys, tokens: Week 79), **generated files** (`__pycache__/`, `*.pyc`, `build/`, `dist/`, `*.egg-info/`, coverage reports), **local environments** (`.venv/`), **editor settings** that are personal, **large data** (keep a small sample and a script that fetches the rest), and the **local database** your program creates. List them in `.gitignore` before the first commit, because removing a secret after it's been pushed means changing it, not just deleting the file.

`.gitignore` patterns, simplified: a plain name (`.env`) matches a file or folder of that name anywhere; a `*` wildcard matches within a name (`*.pyc`); a trailing `/` (`build/`) matches only folders; a leading `/` (`/data`) matches only at the top of the repository; and a leading `!` re-includes something an earlier pattern excluded. The **last** matching pattern wins.

Matching by file name alone is only a first approximation (notice `docs/data/notes.md` isn't really matched by `/data/`); today's exercise does it properly.

## Commits that explain themselves

Commit **small** and **often**: each commit one logical change that leaves the tests passing, so `git bisect` (Week 21) can find the commit that broke something. Write the message in the **Conventional Commits** format:

```
type(scope)!: short description in the imperative

Optional body: why, not what (the diff shows what).

BREAKING CHANGE: what breaks and what to do about it (optional footer)
```

Common types: `feat` (a new feature), `fix` (a bug fix), `docs`, `test`, `refactor` (no behaviour change), `perf`, `build`, `ci` and `chore`. The rules for the next **version** (Week 73's semantic versioning) follow: any breaking change (a `!` or a `BREAKING CHANGE:` footer) bumps MAJOR, any `feat` bumps MINOR, any `fix` or `perf` bumps PATCH, and anything else needs no release. The changelog groups the same messages under **Added**, **Fixed** and **Changed**.

Work on short-lived **branches** and merge them into `main` through pull requests once CI passes, so `main` always works. Even on a solo project, a pull request is where you review your own diff with fresh eyes.

## Continuous integration

**Continuous integration** (Week 74) runs your checks on a clean machine for every push and pull request, so "it works on my computer" stops being the standard. A GitHub Actions workflow is a YAML file in `.github/workflows/`:

```yaml
name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.12", "3.13"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -e ".[dev]"
      - run: ruff check .
      - run: pytest
      - run: python tools/check_architecture.py
```

Checkout, set up Python, install, then the checks from fastest to slowest, so a lint error fails in seconds. The **matrix** runs it on each Python version you support. Add a badge to the README so a broken build is visible, and protect `main` so nothing merges while CI is red.

## A walking skeleton

A **walking skeleton** is the thinnest possible version of the whole system that runs end to end: every module exists, each layer calls the next, a test passes, CI is green. For Waymark: a `cli.py` whose `plan` command calls `services.plan`, which calls `routing.shortest` on a hard-coded two-junction network and prints the result, with one test. It does almost nothing, and that's the point: the hard, boring problems (packaging, imports, CI, the test set-up) are solved on day one, while the code is small, instead of in week five. Tomorrow's first vertical slice puts real work on this skeleton.

## Practice

## Quiz

## Daily challenge

## Your capstone today

On your own computer:

1. **Create the repository** with the layout from the lesson (today's `skeleton` and `write_tree` will write most of it), `git init`, and push it to GitHub. Write `.gitignore` **before** the first commit, including your database file.
2. **Add the documents** from Days 1 to 3 under `docs/`, and `tools/check_architecture.py` from yesterday.
3. **Add CI**: `.github/workflows/ci.yml` like the lesson's. Push, and don't move on until it's green. Add the badge to the README.
4. **Build the walking skeleton**: every module from your architecture exists; the outermost interface calls a service, which calls the domain, with one test that runs the whole chain. Green CI again.
5. **Protect main** (in the repository's settings: require the CI check to pass before merging), and from now on work on branches with Conventional Commit messages.

## Before you go

Clone your repository into a new folder, follow your own README's development instructions exactly, and run the tests. If anything is missing from the instructions, fix the README now. Tomorrow is the **first vertical slice**: one story working properly, through every layer.